## Chức năng chính
File này thực hiện bước **Tiền xử lý dữ liệu & Trích xuất đặc trưng tự động**. Thay vì gán từ khóa thủ công, script này sử dụng các thuật toán thống kê để "học" từ dữ liệu, đảm bảo tính khách quan và độ chính xác cao hơn.

**Quy trình thực hiện:**
1.  **Load Data:** Đọc dữ liệu gốc từ `symptoms.csv`.
2.  **Auto Feature Extraction:** Tự động tìm kiếm các từ khóa (keywords) quan trọng nhất liên quan đến bệnh.
3.  **Auto Weighting:** Tính toán trọng số (độ nghiêm trọng) cho từng từ khóa trên thang điểm [1.0 - 3.0].
4.  **Export:** Xuất ra file từ điển (`diabetes_keywords.json`) và dataset chuẩn (`symptoms_final.csv`) để huấn luyện.

---

## Các thuật toán NLP & Thống kê sử dụng

### 1. TF-IDF (Term Frequency - Inverse Document Frequency)
* **Mục đích:** Vector hóa văn bản, đánh giá tầm quan trọng của từ trong câu.
* **Cơ chế:** Lọc bỏ các từ rác (stopwords) xuất hiện quá nhiều (như "tôi", "là", "thì") và giữ lại các từ mang ý nghĩa đặc trưng (như "sụt cân", "khát").

### 2. Chi-Square Test ($\chi^2$ - Kiểm định Khi bình phương)
* **Mục đích:** Chọn lọc đặc trưng (Feature Selection).
* **Cơ chế:** Đo lường mức độ phụ thuộc giữa **Từ khóa** và **Nhãn bệnh (Outcome)**.
    * Thuật toán sẽ tính toán xem: *"Nếu từ này xuất hiện, xác suất người đó bị bệnh có tăng lên đáng kể không?"*.
    * Giúp loại bỏ các từ ngẫu nhiên không mang tính dự báo.

### 3. N-grams (Bigrams, Trigrams)
* **Mục đích:** Xử lý từ ghép trong tiếng Việt.
* **Cơ chế:** Ghép 2-3 từ liên tiếp để bắt được ngữ nghĩa chính xác hơn từ đơn (VD: "mờ" + "mắt" = "mờ mắt").

In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

print("Đã import thư viện thành công.")

Đã import thư viện thành công.


In [2]:
try:
  df = pd.read_csv('../datasets/raw/symptoms2.csv')
  print(f"File 'symptoms2.csv' đã tải thành công. Gồm {len(df)} dòng.")
except FileNotFoundError:
    print("File 'symptoms2.csv' không tìm thấy.")
    df = pd.DataFrame({'text': [], 'outcome': [], 'stage': []})
    
# Xem 5 dòng đầu
display(df.head()) 

# Xem 5 dòng cuối 
display(df.tail())

File 'symptoms2.csv' đã tải thành công. Gồm 367 dòng.


,text,outcome,stage
0,Tôi bị khát nước và đi tiểu thường xuyên. Tôi ...,1,1
1,Cơ thể tôi run rẩy. Khứu giác và vị giác yếu đ...,1,2
2,"hay thấy ngứa ran ở lòng bàn chân, bàn tay",1,3
3,"tôi thấy nước tiểu có màu hơi hồng, giống như ...",0,0
4,tôi thấy kiến bu quanh nước tiểu của mình ở nh...,1,2


,text,outcome,stage
362,Tôi khó tập trung và cảm xúc thay đổi thất thư...,1,2
363,Tôi cảm thấy cơ bắp run rẩy. Tôi thấy kiệt sức...,1,2
364,tôi ăn chay trường nên không ăn thịt,0,0
365,"Tôi bị ngứa dữ dội, nôn mửa, mệt mỏi và sụt câ...",0,0
366,"Tôi bị sụt cân, cảm thấy thực sự mệt mỏi và nô...",0,0


In [3]:
# Định nghĩa danh sách các từ vô nghĩa (Stop words) tiếng Việt thường gặp
# Những từ này xuất hiện nhiều nhưng không giúp chẩn đoán bệnh
my_stopword = [
    'dạo này', 'gần đây', 'thời gian', 'tôi bị', 'tôi thấy', 'cảm thấy', 
    'không có', 'như thế', 'tôi thường', 'tôi hay','tôi cảm thấy',
    'có phải', 'hay bị', 'triệu chứng', 'người nhà', 'trong người',
    'của tôi', 'là gì', 'như nào', 'rất nhiều', 'mấy hôm', 
    'cho tôi', 'gần đây tôi', 'đặc biệt', 'thường bị', 'thường xuyên'
    'đặc biệt là', 'tôi thường hay', 'tôi hay bị', 'tôi thường bị',
    'mấy ngày', 'mấy hôm nay', 'cả những việc', 'cả những', 'và đau'
    'dù ăn', 'đi chơi', 'tôi đi', 'tôi ăn', 'tôi ngủ', 'tôi uống', 'tôi tập', 
    'tôi làm', 'tôi nghỉ', 'tôi ngồi', 'tôi đứng', 'tôi chạy', 'và', 'tôi', 'bị', 'nhưng'
]

def generate_automatic_keywords(dataframe, top_n=40):
    """
    Sử dụng TF-IDF và Chi-Square để tìm từ khóa quan trọng nhất
    và gán trọng số từ 1.0 đến 3.0
    """
    # Lấy dữ liệu text và nhãn
    texts = dataframe['text'].astype(str).tolist()
    labels = dataframe['outcome'].tolist()

    # Vector hóa bằng TF-IDF (lấy cả từ đơn và từ ghép 2 chữ)
    # min_df=5: Chỉ lấy từ xuất hiện ít nhất 2 lần để tránh nhiễu
    tfidf = TfidfVectorizer(
      ngram_range=(2, 3), 
      max_features=800, 
      min_df=5, 
      stop_words=None
    )
    
    try:
        X_tfidf = tfidf.fit_transform(texts)
        feature_names = tfidf.get_feature_names_out()
    except ValueError:
        print("Dữ liệu quá ít hoặc rỗng, không thể trích xuất keyword.")
        return {}

    # Tính điểm Chi-Square (Độ phụ thuộc giữa từ khóa và bệnh)
    chi2score = chi2(X_tfidf, labels)[0]

    # Ghép từ và điểm số
    wscores = zip(feature_names, chi2score)
    
    # BƯỚC LỌC THỦ CÔNG DANH SÁCH TỪ KHÓA
    filtered_wscores = []
    for word, score in wscores:
        # Chỉ giữ lại từ nếu nó KHÔNG nằm trong my_stopword
        if word not in my_stopword:
            filtered_wscores.append((word, score))
    
    wchi2 = sorted(filtered_wscores, key=lambda x: x[1], reverse=True)
    top_keywords = wchi2[:top_n]
    
    # Chuẩn hóa trọng số về thang điểm [1.0 - 3.0]
    if not top_keywords:
        return {}
        
    min_score = top_keywords[-1][1]
    max_score = top_keywords[0][1]
    
    keywords_dict = {}
    for word, score in top_keywords:
        if max_score == min_score:
            normalized_weight = 1.0
        else:
            normalized_weight = 1 + 2 * ((score - min_score) / (max_score - min_score))
        
        keywords_dict[word] = round(normalized_weight, 1)

    return keywords_dict

print("Hàm trích xuất từ ​​khóa đã được định nghĩa.")

Hàm trích xuất từ ​​khóa đã được định nghĩa.


In [4]:
# Kiểm tra số dòng có NaN trong cột nhãn
missing_outcome = df['outcome'].isna().sum()
print(f"Tổng số dòng có NaN ở cột 'outcome': {missing_outcome}")

missing_stage = df['stage'].isna().sum()
print(f"Tổng số dòng có NaN ở cột 'stage': {missing_stage}")

# In ra tất cả các dòng bị NaN
if missing_outcome > 0:
    print("\n--- Dòng bị NaN ---")
    print(df[df['outcome'].isna()])
else:
    print("Không có dòng nào bị NaN trong cột 'outcome'.")
    
if missing_stage > 0:
    print("\n--- Dòng bị NaN ---")
    print(df[df['stage'].isna()])
else:
    print("Không có dòng nào bị NaN trong cột 'stage'.")


Tổng số dòng có NaN ở cột 'outcome': 0
Tổng số dòng có NaN ở cột 'stage': 0
Không có dòng nào bị NaN trong cột 'outcome'.
Không có dòng nào bị NaN trong cột 'stage'.


In [5]:
print("Đang phân tích dữ liệu...")

# Chạy hàm trích xuất
auto_keywords = generate_automatic_keywords(df, top_n=70) 

# In mẫu kết quả để kiểm tra
print("\n--- TOP 5 TỪ KHÓA QUAN TRỌNG NHẤT ---")
for k, v in list(auto_keywords.items())[:5]:
    print(f"   - '{k}': {v} điểm")

# Lưu ra file JSON
json_filename = '../datasets/raw/diabetes_keywords.json'
with open(json_filename, 'w', encoding='utf-8') as f:
    json.dump(auto_keywords, f, ensure_ascii=False, indent=4)

print(f"\nĐã lưu từ điển trọng số vào '{json_filename}'")

Đang phân tích dữ liệu...

--- TOP 5 TỪ KHÓA QUAN TRỌNG NHẤT ---
   - 'đi tiểu': 3.0 điểm
   - 'đau đầu': 2.8 điểm
   - 'ăn nhiều': 2.7 điểm
   - 'bàn chân': 2.4 điểm
   - 'vết thương': 2.2 điểm

Đã lưu từ điển trọng số vào '../datasets/raw/diabetes_keywords.json'


##Output: file diabetes_keywords.json

Sau khi có file và trọng số (weight), ta lọc thủ công lại lần nữa, chỉ giữ lại các triệu chứng
đặc trưng theo y khoa. Loại bỏ các stopwords khác